**SA234 &#x25aa; Data Wrangling and Visualization &#x25aa; Spring 2026**

# Project 5. COVID-19 in the US, revisited

This project will walk you through the steps necessary to recreate the datasets used in Project 3 from scratch. You will get some practice using the Pandas techniques you've learned so far this semester, learn a few new techniques, and get some exposure to the quirks of working with datasets out in the wild.

For your reference, the datasets from Project 3 have been included in the same folder as this notebook:

1. `data/covid_us_county_20201001.csv` contains data on COVID-19 cases and deaths for each county in the United States as of October 1, 2020. Each row contains the following data for each county:

| Column | Description | 
| :- | :- |
| `date` | Date of observation |
| `county_fips` | County FIPS code |
| `state` | State name |
| `county` | County name |
| `latitude` | Representative latitude coordinate |
| `longitude` | Representative longitude coordinate |
| `population` | Population |
| `cases` | Cumulative total number of cases as of October 1, 2020 |
| `deaths` | Cumulative total number of deaths as of October 1, 2020 |
| `cases_per_100k` | Cumulative total number of cases as of October 1, 2020, per 100,000 people |
| `deaths_per_100k` | Cumulative total number of deaths as of October 1, 2020, per 100,000 people |
| `cases_week` | Number of new cases in the week ending on October 1, 2020 |
| `deaths_week` | Number of new deaths in the week ending on October 1, 2020 |
| `cases_week_per_100k` | Number of new cases in the week ending on October 1, 2020, per 100,000 people |
| `deaths_week_per_100k` | Number of new deaths in the week ending on October 1, 2020, per 100,000 people |


2. `data/covid_us_state_20201001.csv` contains data on COVID-19 cases and deaths for each state in the United States from February 1, 2020 to October 1, 2020. Each row contains the following data for each state and day:

| Column | Description | 
| :- | :- |
| `state` | State name |
| `date` | Date of observation |
| `cases` | Cumulative total number of cases as of October 1, 2020 |
| `deaths` | Cumulative total number of deaths as of October 1, 2020 |
| `cases_day` | Number of new cases on the date of observation |
| `deaths_day` | Number of new deaths on the date of observation |
| `cases_day_per_100k` | Number of cases on the date of observation, per 100,000 people |
| `deaths_day_per_100k` | Number of deaths on the date of observation, per 100,000 people |

Before we start, let's import Pandas.

In [ ]:
import pandas as pd

<hr style="border-top: 2px solid gray; margin-top: 1px; margin-bottom: 1px"></hr>

## Problem 1 &mdash; Population data

To compute the per capita metrics, such as `cases_per_100k` in `data/covid_us_county_20201001.csv`, we will use the 2019 estimated county populations from the US Census.
In the same folder as this notebook, there is a CSV file named `data/co-est2019-alldata.csv` containing the estimated county populations from April, 1 2010 to July 1, 2019.

1. Open the CSV file in Excel to get a sense of the dataset. (Note that you'll probably have trouble opening the file in JupyterLab due to encoding issues. We'll take care of this shortly.) If Excel asks if it should convert the data in order to remove the leading zeros, select *Don't Convert*. We'll focus on the following columns:

    | Column | Description |
    | :- | :- |
    | `STATE` | 2-digit state FIPS code |
    | `COUNTY` | 3-digit county FIPS code |
    | `STNAME` | State name | 
    | `CTYNAME` | County name |
    | `POPESTIMATE2019` | Estimated total population as of 7/1/2019 |

2. Using a single method chain, read and wrangle this dataset, and put the results into a DataFrame called `county_population_df`:
    1. Use `pd.read_csv()` to read in the file.  Due to some quirks in how the file is formatted, you'll need to use the `encoding='ISO-8859-1'` and `engine='python'` keyword arguments. Make sure to read the state and county FIPS codes as strings.
    2. Remove the rows whose 3-digit county FIPS code is equal to the string `"000"`. These rows contain state-wide totals.
    3. Create a new column called `county_fips` containing a "full" 5-digit county FIPS code by concatenating the strings in `STATE` and `COUNTY`. Use the [`.str.cat()` Series method](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.cat.html). In particular, if you have a DataFrame `x` with string columns named `first` and `second`, then you can concatenate the strings in these columns like this:<br><br>
        ```python
        x['first'].str.cat(x['second'])
        ```
        <br>
    5. Keep only the following columns: `county_fips`, `STNAME`, `CTYNAME`, and `POPESTIMATE2019`. We won't need the others.
    6. Rename the columns: `STNAME` &rarr; `state`, `CTYNAME` &rarr; `county`, `POPESTIMATE2019` &rarr; `population`.
    7. Reset the index.

  
3. Display the first 5 rows of `county_population_df`. You should see something like this:

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>county_fips</th>
      <th>state</th>
      <th>county</th>
      <th>population</th>
    </tr>
  </thead>
  <tbody>
    <tr style="text-align: right;">
      <th>0</th>
      <td>01001</td>
      <td>Alabama</td>
      <td>Autauga County</td>
      <td>55869</td>
    </tr>
    <tr style="text-align: right;">
      <th>1</th>
      <td>01003</td>
      <td>Alabama</td>
      <td>Baldwin County</td>
      <td>223234</td>
    </tr>
    <tr style="text-align: right;">
      <th>2</th>
      <td>01005</td>
      <td>Alabama</td>
      <td>Barbour County</td>
      <td>24686</td>
    </tr>
    <tr style="text-align: right;">
      <th>3</th>
      <td>01007</td>
      <td>Alabama</td>
      <td>Bibb County</td>
      <td>22394</td>
    </tr>
    <tr style="text-align: right;">
      <th>4</th>
      <td>01009</td>
      <td>Alabama</td>
      <td>Blount County</td>
      <td>57826</td>
    </tr>
  </tbody>
</table>

## Problem 2 &mdash; Geographic data

To obtain the latitude and longitude of each county, as in the CSV file `data/covid_us_county_20201001.csv`, we will again use data from the US Census.
In the same folder as this notebook, there is a text file called `data/2019_Gaz_counties_national.txt`, which contains the 2019 US National Counties Gazetteer File.

1. Open the file in Excel as a *tab-delimited* file to get a sense of the dataset. Again, if Excel asks if it should convert the data in order to remove the leading zeros, select *Don't Convert*. We'll focus on the following columns:

    | Column | Description |
    | :- | :- |
    | `GEOID` | "Full" county FIPS code |
    | `INTPTLAT` | Representative latitude coordinates |
    | `INTPTLONG` | Representative longitude coordinates |

    In Excel, note that the name of `INPTLONG` has a lot of *trailing whitespace* (extra spaces at the end). You'll take care of this when we read in the file with Pandas.

2. Using a single method chain, read and wrangle this dataset, and put the results into a DataFrame called `county_location_df`:
    1. Use `pd.read_csv()` to read in the file.  You can read tab-delimited files with `pd.read_csv()` using the `delimiter='\t'` keyword argument. Make sure to read the county FIPS codes as strings.
    2. Sanitize the column names by removing all leading and trailing whitespace (extra spaces at the beginning and the end). You can do this with the `.rename()` DataFrame method. Instead of passing a dictionary to the keyword argument `columns=...`, you can pass a *function* that acts on each column name string. In this case, we can use the [`.strip()` Python string method](https://docs.python.org/3/library/stdtypes.html#str.strip) to remove leading and trailing whitespace from each column name, like this:<br><br>
        ```python
        .rename(columns=lambda s: s.strip())
        ```
        <br>
   3. Keep only the following columns: `GEOID`, `INTPTLAT`, `INTPTLONG`. We won't need the others.
   4. Rename the columns: `GEOID` &rarr; `county_fips`, `INTPTLAT` &rarr; `latitude`, `INTPTLONG` &rarr; `longitude`.  From now on, we'll rename any column containing the county FIPS codes to `county_fips`, to make merging DataFrames easier later on.

3. Display the first 5 rows of `county_location_df`. You should see something like this:

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>county_fips</th>
      <th>latitude</th>
      <th>longitude</th>
    </tr>
  </thead>
  <tbody>
    <tr style="text-align: right;">
      <th>0</th>
      <td>01001</td>
      <td>32.532237</td>
      <td>-86.646440</td>
    </tr>
    <tr style="text-align: right;">
      <th>1</th>
      <td>01003</td>
      <td>30.659218</td>
      <td>-87.746067</td>
    </tr>
    <tr style="text-align: right;">
      <th>2</th>
      <td>01005</td>
      <td>31.870253</td>
      <td>-85.405104</td>
    </tr>
    <tr style="text-align: right;">
      <th>3</th>
      <td>01007</td>
      <td>33.015893</td>
      <td>-87.127148</td>
    </tr>
    <tr style="text-align: right;">
      <th>4</th>
      <td>01009</td>
      <td>33.977358</td>
      <td>-86.566440</td>
    </tr>
  </tbody>
</table>

## Problem 3 &mdash; Number of cases

To obtain the number of COVID-19 cases in each county, we will use data from [USAFacts](https://usafacts.org/). In the same folder as this notebook, there is a CSV file named `data/covid_confirmed_usafacts_2020.csv` containing the cumulative number of COVID-19 cases in each county from January 22, 2020 to December 31, 2020.

1. Open the CSV file in Excel to get a sense of the dataset. Again, if Excel asks if it should convert the data in order to remove the leading zeros, select *Don't Convert*. We will focus on the following columns:
    
    | Column | Description |
    | :- | :- |
    | `countyFIPS` | County FIPS code |
    | `yyyy-mm-dd` | Number of cases on date yyyy-mm-dd |

    Note that the county FIPS codes in this dataset are not all five digits - we'll fix this shortly.
   

2. Using a single method chain, read and wrangle this dataset and put the results into a DataFrame called `cases_df`:
    1. Use `pd.read_csv()` to read in the file. Make sure to read the county FIPS codes as strings.
    2. Create a new column called `county_fips` that adds leading zeros to `countyFIPS` so that they are all five digits using the [`str.zfill()` Series method](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.zfill.html). 
    3. Drop the following columns: `countyFIPS`, `StateFIPS`, `State`, and `County Name`. We don't need them.

3. Display the first 5 rows of `cases_df`. You should see something like this:

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>2020-01-22</th>
      <th>2020-01-23</th>
      <th>2020-01-24</th>
      <th>...</th>
      <th>2020-12-30</th>
      <th>2020-12-31</th>
      <th>county_fips</th>
    </tr>
  </thead>
  <tbody>
    <tr style="text-align: right;">
      <th>0</th>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>...</td>
      <td>4164</td>
      <td>4190</td>
      <td>01001</td>
    </tr>
    <tr style="text-align: right;">
      <th>1</th>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>...</td>
      <td>13392</td>
      <td>13601</td>
      <td>01003</td>
    </tr>
    <tr style="text-align: right;">
      <th>2</th>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>...</td>
      <td>1492</td>
      <td>1514</td>
      <td>01005</td>
    </tr>
    <tr style="text-align: right;">
      <th>3</th>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>...</td>
      <td>1817</td>
      <td>1834</td>
      <td>01007</td>
    </tr>
    <tr style="text-align: right;">
      <th>4</th>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>...</td>
      <td>4584</td>
      <td>4641</td>
      <td>01009</td>
    </tr>
  </tbody>
</table>

## Problem 4 &mdash; Number of cases, cont.

Let's tidy the data in `cases_df` by pivoting it to long form, so that the dates of observation are in a column.

1. Using a single method chain, create a new DataFrame `cases_long_df`:
    1. Pivot `cases_df` from wide to long form. Put the dates into a column named `date`, and the numbers of cases into a column named `cases`.
    2. Overwrite the existing `date` column using `.assign()` with the same date values, converted to proper Python datetime objects with [the `pd.to_datetime()` function](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html).

2. Display the first 5 rows of `cases_long_df`. You should see something like this:

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>county_fips</th>
      <th>date</th>
      <th>cases</th>
    </tr>
  </thead>
  <tbody>
    <tr style="text-align: right;">
      <th>0</th>
      <td>01001</td>
      <td>2020-01-22</td>
      <td>0</td>
    </tr>
    <tr style="text-align: right;">
      <th>1</th>
      <td>01003</td>
      <td>2020-01-22</td>
      <td>0</td>
    </tr>
    <tr style="text-align: right;">
      <th>2</th>
      <td>01005</td>
      <td>2020-01-22</td>
      <td>0</td>
    </tr>
    <tr style="text-align: right;">
      <th>3</th>
      <td>01007</td>
      <td>2020-01-22</td>
      <td>0</td>
    </tr>
    <tr style="text-align: right;">
      <th>4</th>
      <td>01009</td>
      <td>2020-01-22</td>
      <td>0</td>
    </tr>
  </tbody>
</table>

## Problem 5 &mdash; Number of deaths

To obtain the number of COVID-19 deaths in each county, we will again use data from USAFacts. In the same folder as this notebook, there is a CSV file named `data/covid_deaths_usafacts_2020.csv` containing the cumulative number of COVID-19 deaths in each county from January 22, 2020 to December 31, 2020.

Use the same instructions as in Problem 3 to read and wrangle this dataset. Put the results in a DataFrame called `deaths_df`. Display the first 5 rows of `deaths_df`.

## Problem 6 &mdash; Number of deaths, cont.

Use the same instructions as in Problem 4 to tidy the deaths data from USAFacts. Put the results into a DataFrame called `deaths_long_df`, with the number of deaths in a column called `deaths`. Display the first 5 rows of `deaths_long_df`.

## Problem 7 &mdash; Merge the data together

Now it's finally time to merge the population data, geographic data, and COVID-19 cases data. 

1. Using a single method chain, merge the datasets together and put the results in a DataFrame called `merged_df`:<br>
    1. Start with `cases_long_df` as the "left" DataFrame and `deaths_long_df` as the "right" DataFrame. Merge on `county_fips` and `date`, using the keys from the "left" only.
    2. Take the resulting DataFrame from part A as the "left", and `county_location_df` as the "right". Merge on `county_fips`, using keys from the "left" only.
    3. Take the resulting DataFrame from part B as the "left", and `county_population_df` as the "right". Merge on `county_fips`, using keys from the "left" only.

2. Display the first 5 rows of `merged_df`. You should see something like this: 

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>county_fips</th>
      <th>date</th>
      <th>cases</th>
      <th>deaths</th>
      <th>latitude</th>
      <th>longitude</th>
      <th>state</th>
      <th>county</th>
      <th>population</th>
    </tr>
  </thead>
  <tbody>
    <tr style="text-align: right;">
      <th>0</th>
      <td>01001</td>
      <td>2020-01-22</td>
      <td>0</td>
      <td>0</td>
      <td>32.532237</td>
      <td>-86.646440</td>
      <td>Alabama</td>
      <td>Autauga County</td>
      <td>55869</td>
    </tr>
    <tr style="text-align: right;">
      <th>1</th>
      <td>01003</td>
      <td>2020-01-22</td>
      <td>0</td>
      <td>0</td>
      <td>30.659218</td>
      <td>-87.746067</td>
      <td>Alabama</td>
      <td>Baldwin County</td>
      <td>223234</td>
    </tr>
    <tr style="text-align: right;">
      <th>2</th>
      <td>01005</td>
      <td>2020-01-22</td>
      <td>0</td>
      <td>0</td>
      <td>31.870253</td>
      <td>-85.405104</td>
      <td>Alabama</td>
      <td>Barbour County</td>
      <td>24686</td>
    </tr>
    <tr style="text-align: right;">
      <th>3</th>
      <td>01007</td>
      <td>2020-01-22</td>
      <td>0</td>
      <td>0</td>
      <td>33.015893</td>
      <td>-87.127148</td>
      <td>Alabama</td>
      <td>Bibb County</td>
      <td>22394</td>
    </tr>
    <tr style="text-align: right;">
      <th>4</th>
      <td>01009</td>
      <td>2020-01-22</td>
      <td>0</td>
      <td>0</td>
      <td>33.977358</td>
      <td>-86.566440</td>
      <td>Alabama</td>
      <td>Blount County</td>
      <td>57826</td>
    </tr>
  </tbody>
</table>

## Problem 8 &mdash; Compute additional metrics

Now that we have a DataFrame that contains the population, geographic, and COVID-19 data, we can compute some additional metrics.

1. Using a single method chain, produce a new DataFrame called `df` with the following new metrics:
    1. **Cases per 100,000.** Create a new variable called `cases_per_100k` containing the number of cases per 100,000 for each observation (i.e., each `county_fips`-`date` pair). Use the following formula:
        $$
        \text{number of cases per 100000} = \frac{\text{number of cases}}{\text{population}} \times \text{100000}
        $$
    3. **Deaths per 100,000.** Create a new variable called `deaths_per_100k` containing the number of deaths per 100,000 for each observation.
    4. **New cases in the last week.** To do this, first make sure `merged_df` is sorted by `county_fips` and `date`. Then create a new variable called `cases_week` using split-apply-combine: group the data by `county_fips`, then use `.transform()` to apply `.diff(7)` to each group's `cases` column. [Here is the documentation for `.diff()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.diff.html).
    5. **New deaths in the last week.** Do the same as in part C, except call the new variable `deaths_week`, and perform split-apply-transform on the `deaths` column.
    6. **New cases in the last week per 100,000.** Create a new variable called `cases_week_per_100k` containing the number of new cases in the last week per 100,000 for each observation.
    7. **New deaths in the last week per 100,000.** Create a new variable called `deaths_week_per_100k` containing the number of deaths in the last week per 100,000 for each observation.

2. Reset the index.

3. Display the **last** 5 rows of `df` using `.tail()`, so you can see if `cases_week` and `deaths_week` were computed correctly. (The first 5 rows are not that interesting.) You should see something that looks like this:

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>county_fips</th>
      <th>date</th>
      <th>cases</th>
      <th>deaths</th>
      <th>latitude</th>
      <th>longitude</th>
      <th>state</th>
      <th>county</th>
      <th>population</th>
      <th>cases_per_100k</th>
      <th>deaths_per_100k</th>
      <th>cases_week</th>
      <th>deaths_week</th>
      <th>cases_week_per_100k</th>
      <th>deaths_week_per_100k</th>
    </tr>
  </thead>
  <tbody>
    <tr style="text-align: right;">
      <th>1083985</th>
      <td>56045</td>
      <td>2020-12-27</td>
      <td>466</td>
      <td>2</td>
      <td>43.846213</td>
      <td>-104.57002</td>
      <td>Wyoming</td>
      <td>Weston County</td>
      <td>6927</td>
      <td>6727.298975</td>
      <td>28.872528</td>
      <td>20.0</td>
      <td>0.0</td>
      <td>288.725278</td>
      <td>0.0</td>
    </tr>
    <tr style="text-align: right;">
      <th>1083986</th>
      <td>56045</td>
      <td>2020-12-28</td>
      <td>467</td>
      <td>2</td>
      <td>43.846213</td>
      <td>-104.57002</td>
      <td>Wyoming</td>
      <td>Weston County</td>
      <td>6927</td>
      <td>6741.735239</td>
      <td>28.872528</td>
      <td>20.0</td>
      <td>0.0</td>
      <td>288.725278</td>
      <td>0.0</td>
    </tr>
    <tr style="text-align: right;">
      <th>1083987</th>
      <td>56045</td>
      <td>2020-12-29</td>
      <td>471</td>
      <td>2</td>
      <td>43.846213</td>
      <td>-104.57002</td>
      <td>Wyoming</td>
      <td>Weston County</td>
      <td>6927</td>
      <td>6799.480294</td>
      <td>28.872528</td>
      <td>16.0</td>
      <td>0.0</td>
      <td>230.980222</td>
      <td>0.0</td>
    </tr>
    <tr style="text-align: right;">
      <th>1083988</th>
      <td>56045</td>
      <td>2020-12-30</td>
      <td>475</td>
      <td>2</td>
      <td>43.846213</td>
      <td>-104.57002</td>
      <td>Wyoming</td>
      <td>Weston County</td>
      <td>6927</td>
      <td>6857.225350</td>
      <td>28.872528</td>
      <td>15.0</td>
      <td>0.0</td>
      <td>216.543958</td>
      <td>0.0</td>
    </tr>
    <tr style="text-align: right;">
      <th>1083989</th>
      <td>56045</td>
      <td>2020-12-31</td>
      <td>476</td>
      <td>2</td>
      <td>43.846213</td>
      <td>-104.57002</td>
      <td>Wyoming</td>
      <td>Weston County</td>
      <td>6927</td>
      <td>6871.661614</td>
      <td>28.872528</td>
      <td>16.0</td>
      <td>0.0</td>
      <td>230.980222</td>
      <td>0.0</td>
    </tr>
  </tbody>
</table>

## Problem 9 &mdash; County-level data on October 1

Now we're ready to recreate `data/covid_us_county_20201001.csv`.

1. Using a single method chain, create a new DataFrame called `county_oct1_df` based on `df`:
    1. Keep only the rows corresponding to October 1, 2020. In order to use dates with `.query()`, you need to first import the `datetime` library. Then you can check if the `date` variable is equal to October 1, 2020 like this:<br><br>
        ```python
        .query('date == datetime.date(2020, 10, 1)')
        ```
        <br>
    2. Sort the rows by `county_fips`.
    3. Reorder the columns in this order: `date`, `county_fips`, `state`, `county`, `latitude`, `longitude`, `population`, `cases`, `deaths`, `cases_per_100k`, `deaths_per_100k`, `cases_week`, `deaths_week`, `cases_week_per_100k`, `deaths_week_per_100k`

2. Write this DataFrame to a CSV file called `county_oct1_df.csv`. You can do this with the `.to_csv()` DataFrame method, like the code snippet below.  

    ```python
    county_oct1_df.to_csv('county_oct1_df.csv', index=False)
    ```
    <br>
   The `index=False` keyword argument prevents the index being included in the output file. 

## Problem 10 &mdash; State-level data between February 1 and October 1

To recreate `data/covid_us_state_20201001.csv`, we'll need to do a little more work.

1. Using a single method chain, create a new DataFrame called `state_feb1_oct1_df` that contains the new daily cases and deaths in each state between February 1 and October 1:
    1. Keep only the rows of `df` corresponding to dates between January 31, 2020 and October 1, 2020, inclusive. See Problem 9 for guidance on how to use dates with `.query()`. Instead of `==`, you can use `<=` and `>=`.
    2. Use split-apply-combine to create a DataFrame that contains the total number of cases, the total number of deaths, and the total population for each state on each date. Name these columns `cases`, `deaths`, and `population`, respectively.
    3. Ensure that the resulting DataFrame is sorted by `state` and `date`.
    4. Use split-apply-combine to create a new variable called `cases_day` containing the number of new cases in the past day: group the data by `state` (but not `date`), then use `.transform()` to apply `.diff(1)` to each group's `cases` column.
    5. Do the same for deaths, creating a new variable called `deaths_day` containing the number of new deaths in the past day.
    6. Create a new variable called `cases_day_per_100k` containing the number of cases per 100,000.
    7. Create a new variable called `deaths_day_per_100k` containing the number of deaths per 100,000.
    8. Drop any row with missing data.
    9. Keep only the following columns, in this order: `state`, `date`, `cases`, `deaths`, `cases_day`, `deaths_day`, `cases_day_per_100k`, `deaths_day_per_100k`.

2. Write this DataFrame to a CSV file called `state_feb1_oct1_df.csv`.    

## Problem 11 &mdash; Check your work on the county-level data

In the same folder as this notebook, there is a Python file named `compare.py` that contains a function called `compare_csv()` that compares the contents of two CSV files.

The code below uses this function to compare the CSV file you generated in Problem 9 with the corresponding CSV file from Project 3 (`data/covid_us_county_20201001.csv`). 

Run the cell below. Your score for this problem will depend on the fraction of matching values between your CSV file and the CSV file from Project 3.

In [ ]:
from compare import compare_csv

compare_csv('county_oct1_df.csv', 'data/covid_us_county_20201001.csv')

## Problem 12 &mdash; Check your work on the state-level data

The code below uses this function to compare the CSV file you generated in Problem 10 with the corresponding CSV file from Project 3 (located in `data/covid_us_state_20201001.csv`). 

Run the cell below. Your score for this problem will depend on the fraction of matching values between your CSV file and the CSV file from Project 3.

In [ ]:
compare_csv('state_feb1_oct1_df.csv', 'data/covid_us_state_20201001.csv')

<hr style="border-top: 2px solid gray; margin-top: 1px; margin-bottom: 1px"></hr>

## Grading rubric

| Problem |                                                     | Points  |
| :-      | :-                                                  | -:      |
| 1       | 2A Read CSV file                                    | 1       |
|         | 2B Delete specified rows                            | 1       |
|         | 2C Create full county FIPS codes                    | 1       |
|         | 2D Keep specified columns                           | 1       |
|         | 2E Rename columns as specified                      | 1       |
|         | 2F Reset the index                                  | 1       |
|         | 3 Display first 5 rows                              | 1       |
|         | Uses a single method chain                          | 1       |
|         | Code runs without errors                            | 2       |
| 2       | 2A Read tab-delimited file                          | 1       |
|         | 2B Remove white space before and after column names | 1       |
|         | 2C Keep specified columns                           | 1       |
|         | 2D Rename columns as specified                      | 1       |
|         | 3 Display first 5 rows                              | 1       |
|         | Uses a single method chain                          | 1       |
|         | Code runs without errors                            | 1       |
| 3       | 2A Read CSV file                                    | 1       |
|         | 2B Create new column with 5-digit county FIPS codes | 1       | 
|         | 2C Drop specified columns                           | 1       |
|         | 3 Display first 5 rows                              | 1       |
|         | Uses a single method chain                          | 1       |
|         | Code runs without errors                            | 1       |
| 4       | 1A Pivot from wide to long form as specified        | 2       |
|         | 1B Convert dates to datetime objects                | 1       |
|         | 2 Display first 5 rows                              | 1       |
|         | Uses a single method chain                          | 1       |
|         | Code runs without errors                            | 2       |
| 5       | 2A Read CSV file                                    | 1       |
|         | 2B Create new column with 5-digit county FIPS codes | 1       |
|         | 2C Drop specified columns                           | 1       |
|         | 3 Display first 5 rows                              | 1       |
|         | Uses a single method chain                          | 1       |
|         | Code runs without errors                            | 1       |
| 6       | 1A Pivot from wide to long form as specified        | 2       |
|         | 1B Convert dates to datetime objects                | 1       |
|         | 2 Display first 5 rows                              | 1       |
|         | Uses a single method chain                          | 1       |
|         | Code runs without errors                            | 2       |
| 7       | 1ABC Merge DataFrames as specified                  | 4       |
|         | 2 Display first 5 rows                              | 1       |
|         | Uses a single method chain                          | 1       |
|         | Code runs without errors                            | 2       |
| 8       | 1A Compute cases per 100,000                        | 1       |
|         | 1B Compute deaths per 100,000                       | 1       |
|         | 1C Compute new cases in last week                   | 2       |
|         | 1D Compute new deaths in last week                  | 2       |
|         | 1E Compute new cases in last week per 100,000       | 1       |
|         | 1F Compute new deaths in last week per 100,000      | 1       |
|         | 2 Reset the index                                   | 1       |
|         | 3 Display last 5 rows                               | 1       |
|         | Uses a single method chain                          | 1       |
|         | Code runs without errors                            | 2       |
| 9       | 1A Keep rows as specified                           | 1       |
|         | 1B Sort rows as specified                           | 1       |
|         | 1C Reorder columns as specified                     | 1       |
|         | 2 Write DataFrame to CSV file                       | 2       |
|         | Uses a single method chain                          | 1       |
|         | Code runs without errors                            | 2       |
| 10      | 1A Keep rows as specified                           | 1       |
|         | 1B Compute cases, deaths, population by state       | 2       |
|         | 1C Sort by state and date                           | 1       |
|         | 1D Compute cases per day by state                   | 2       |
|         | 1E Compute deaths per day by state                  | 2       |
|         | 1F Compute cases per day per 100,000 by state       | 1       |
|         | 1G Compute deaths per day per 100,000 by state      | 1       |
|         | 1H Drop rows with missing data                      | 1       |
|         | 1I Keep and order columns as specified              | 1       |
|         | 2 Write DataFrame to CSV file                       | 2       |
|         | Uses a single method chain                          | 1       |
|         | Code runs without errors                            | 3       |
| 11      | County-level CSV file matches Project 3 CSV file    | 5       |
| 12      | State-level CSV file matches Project 3 CSV file     | 5       |
|         | **Total**                                           | **100** |